In [ ]:
# Data file: set CMSOPEN_DATA_FILE to the absolute path of a NanoAOD ROOT file
# before starting Jupyter, e.g. the 2012C DoubleMuParked file described in DATA.md.
import os
from pathlib import Path

if not os.environ.get("CMSOPEN_DATA_FILE"):
    raise RuntimeError(
        "CMSOPEN_DATA_FILE is not set. Start Jupyter with "
        "CMSOPEN_DATA_FILE=/absolute/path/to/file.root jupyter lab ..."
    )
DATA_FILE = Path(os.environ["CMSOPEN_DATA_FILE"])
if not DATA_FILE.exists():
    raise FileNotFoundError(f"CMSOPEN_DATA_FILE points to a missing file: {DATA_FILE}")
print(f"Data file: {DATA_FILE}")

# Layer 1: reading CMS NanoAOD with `cmsopen.ingestion`

A NanoAOD file stores one row per collision *event* and one column (branch) per quantity. Particle quantities such as `Muon_pt` are *jagged*: each event holds a list whose length is that event's number of muons. `open_events` opens the file lazily: only metadata is read until we ask for specific branches.

In [ ]:
import numpy as np

import cmsopen
from cmsopen import open_events
from cmsopen.ingestion import DIMUON_TRIGGERS_2012

print(f"cmsopen imported from: {cmsopen.__file__}")
events = open_events(DATA_FILE)
print(events.summary())

## Raw muons in the first three events

`pt` is the transverse momentum in GeV and `charge` is ±1. No cuts are applied, so soft and poorly measured muons appear alongside the energetic ones.

In [ ]:
muons = events.muons(entry_stop=3)
for i in range(3):
    print(f"event {i}: Muon_pt     = {muons.pt[i].to_numpy().astype(float).round(5).tolist()}")
    print(f"         Muon_charge = {muons.charge[i].tolist()}")

## Dimuon trigger fraction

The High Level Trigger decides online which collisions to keep. An event counts as triggered if either 2012 double-muon path fired.

In [ ]:
N_TRIGGER_EVENTS = 100_000
fired = events.trigger(*DIMUON_TRIGGERS_2012, entry_stop=N_TRIGGER_EVENTS)
fraction = float(fired.mean())
print(f"Dimuon trigger ({' OR '.join(DIMUON_TRIGGERS_2012)}) fraction, "
      f"first {len(fired):,} events: {fraction:.4f}")

## Self-checks

These compare against numbers measured on the 2012C DoubleMuParked file (`45F7AE81-7A79-4CB1-998E-7BD156490294.root`). For any other file they are skipped.

In [ ]:
class Checks:
    """Collects self-check results and prints one OK/FAIL/SKIP line each."""

    def __init__(self):
        self.failures = []

    def check(self, name, ok, detail):
        print(f"{'OK' if ok else 'FAIL'}: {name} ({detail})")
        if not ok:
            self.failures.append(name)

    def skip(self, name, reason):
        print(f"SKIP: {name} ({reason})")

In [ ]:
KNOWN_FILE = "45F7AE81-7A79-4CB1-998E-7BD156490294.root"
KNOWN_N_EVENTS = 396_766
KNOWN_FIRST_PTS = [71.24144, 25.51886, 3.74997]
KNOWN_TRIGGER_FRACTION = 0.5212

checks = Checks()
if DATA_FILE.name != KNOWN_FILE:
    reason = f"{DATA_FILE.name} is not the known 2012C file {KNOWN_FILE}"
    checks.skip("event count", reason)
    checks.skip("first-event muon pts", reason)
    checks.skip("dimuon trigger fraction", reason)
else:
    checks.check("event count", len(events) == KNOWN_N_EVENTS, f"{len(events):,}, expected {KNOWN_N_EVENTS:,}")
    pts = muons.pt[0].to_numpy().astype(float)
    checks.check(
        "first-event muon pts",
        len(pts) == len(KNOWN_FIRST_PTS) and np.allclose(pts, KNOWN_FIRST_PTS, rtol=0, atol=1e-5),
        f"{pts.round(5).tolist()}, expected {KNOWN_FIRST_PTS}",
    )
    checks.check(
        "dimuon trigger fraction",
        abs(fraction - KNOWN_TRIGGER_FRACTION) <= 0.002,
        f"{fraction:.4f}, expected {KNOWN_TRIGGER_FRACTION} +/- 0.002",
    )
events.close()

if checks.failures:
    raise AssertionError(f"{len(checks.failures)} check(s) failed: {', '.join(checks.failures)}")
print("All checks passed")